# 08 — Bundles and Cross-Trace Comparison

**Purpose:** Show how to group multiple `Trace` objects into a `Bundle`, align them
structurally, read per-layer cross-trace activations via the Super* accessor family,
measure divergence with `diff_pair` / `most_changed`, and sweep a replacement value
across runs. The canonical use-case: compare a clean pass vs. an ablated pass and
identify which layers changed most.

**Surfaces covered:**
- [ ] `tl.bundle([...])` — construct a Bundle from a sequence of Traces
- [ ] `Bundle.names` / `Bundle.members` — member enumeration
- [ ] `Bundle.shared_layer_labels` / `Bundle.divergent_layer_labels` — structural alignment
- [ ] `Bundle.is_structurally_consistent` — structural check
- [ ] `Bundle.relationship(a, b)` — infer how two members relate
- [ ] `Bundle.layers` — `SuperLayerAccessor` (Super* accessor family)
- [ ] `Bundle.ops` — `SuperOpAccessor`
- [ ] `Bundle.modules` — `SuperModuleAccessor`
- [ ] `Bundle.params` — `SuperParamAccessor`
- [ ] `SuperOp` — per-label cross-trace record (`repr`, `.out`, `.members`, `.shape`)
- [ ] `SuperOp.aggregate(statistic=...)` — stack/reduce across traces
- [ ] `SuperOp.diff_pair(metric=...)` — pairwise distance matrix
- [ ] `Bundle.at(label)` — shorthand accessor returning a `SuperOp`
- [ ] `Bundle.diff_pair(a, b)` — per-layer divergence list
- [ ] `Bundle.most_changed(baseline=...)` — rank layers by change from baseline
- [ ] `tl.sweep(model, x, param, values, names=...)` — one-trace-per-value Bundle factory
- [ ] `Bundle.save(path)` / `tl.load(path)` — round-trip a Bundle to disk
- [ ] `Bundle.__repr__` — ergonomics check

## 1. Setup

In [ ]:
import pathlib
import sys
import tempfile

sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torchlens as tl
from _models import ZOO

print(f"torchlens : {tl.__version__}")
print(f"torch     : {torch.__version__}")

## 2. Build two traces — clean pass and ablated pass

We use `tiny_mlp` for all bundle demos: same model, same weights, different inputs.
One trace is the "clean" baseline; the other represents an ablated or perturbed input.
This is the canonical use-case for `Bundle`.

In [ ]:
# _vocab: tl.bundle, tl.Bundle, Bundle.names, Bundle.members

torch.manual_seed(0)
model, x_clean = ZOO["tiny_mlp"]()
x_ablated = torch.randn(2, 8)

trace_clean = tl.trace(model, x_clean)
trace_ablated = tl.trace(model, x_ablated)

print("clean trace   :", repr(trace_clean))
print("ablated trace :", repr(trace_ablated))

## 3. Constructing a Bundle

`tl.bundle(members)` accepts a list of `(name, Trace)` pairs (or a plain list for
auto-naming). Pass named pairs so diff / most-changed references are readable.

In [ ]:
b = tl.bundle([("clean", trace_clean), ("ablated", trace_ablated)])

print("type         :", type(b).__name__)
print("names        :", b.names)
print()
print("members dict :", b.members)
print()
print("repr(b)      :", repr(b))  # <- ergonomics check: no __repr__ yet

## 4. Structural alignment — shared vs divergent labels

In [ ]:
# _vocab: Bundle.shared_layer_labels, Bundle.divergent_layer_labels,
#         Bundle.is_structurally_consistent, Bundle.relationship

print("shared_layer_labels   :", b.shared_layer_labels)
print("divergent_layer_labels:", b.divergent_layer_labels)
print("is_structurally_consistent:", b.is_structurally_consistent)
print()
rel = b.relationship("clean", "ablated")
print("relationship(clean, ablated):", rel)
print("  (same model object, different input -> SHARED_GRAPH_DIFFERENT_INPUT expected)")

## 5. Super* accessors — cross-trace views of each layer

Each bundle accessor (`b.layers`, `b.ops`, `b.modules`, `b.params`) returns a
**Super* accessor** that aligns the same label across all member traces.
Indexing it by label returns a `SuperOp` (or `SuperLayer` alias) record.

In [ ]:
# _vocab: Bundle.layers (SuperLayerAccessor), Bundle.ops (SuperOpAccessor),
#         Bundle.modules (SuperModuleAccessor), Bundle.params (SuperParamAccessor)

print("b.layers  :", repr(b.layers))
print("b.ops     :", repr(b.ops))
print("b.modules :", repr(b.modules))
print("b.params  :", repr(b.params))

In [ ]:
# Index by label to get a SuperOp — cross-trace record for that layer
# _vocab: SuperOp, SuperOp.label, SuperOp.members, SuperOp.out, SuperOp.shape

super_relu = b.layers["relu_1_2"]

print("type        :", type(super_relu).__name__)
print("repr        :", repr(super_relu))
print("label       :", super_relu.label)
print("members     :", super_relu.members)
print("shape       :", super_relu.shape)
print()
print("out (stacked):", super_relu.out)
print("  (rows = batch x members; .out concatenates across traces)")

## 6. `bundle.at(label)` — shorthand for a single SuperOp

In [ ]:
# _vocab: Bundle.at

at_relu = b.at("relu_1_2")
print("b.at('relu_1_2') type :", type(at_relu).__name__)
print("b.at('relu_1_2') repr :", repr(at_relu))
print("same as b.layers['relu_1_2']:", at_relu.label == super_relu.label)

## 7. Per-layer aggregate and diff_pair

`SuperOp.aggregate(statistic=)` reduces across the member axis.  
`SuperOp.diff_pair(metric=)` returns a pairwise NxN distance matrix.

In [ ]:
# _vocab: SuperOp.aggregate, SuperOp.diff_pair

print("aggregate(statistic='mean') shape:", super_relu.aggregate(statistic="mean").shape)
print("aggregate(statistic='std')  shape:", super_relu.aggregate(statistic="std").shape)
print()
print("diff_pair (cosine) [N x N matrix]:\n", super_relu.diff_pair(metric="cosine"))
print()
print("diff_pair (relative_l2):\n", super_relu.diff_pair(metric="relative_l2"))

## 8. Bundle-level diff: `diff_pair` and `most_changed`

`Bundle.diff_pair(a, b)` returns a **list of (label, score)** pairs across all shared layers.
`Bundle.most_changed(baseline=)` ranks them descending by divergence from the baseline trace.

In [ ]:
# _vocab: Bundle.diff_pair, Bundle.most_changed

layer_diffs = b.diff_pair("clean", "ablated")
print("diff_pair (clean vs ablated):")
for label, score in layer_diffs:
    bar = "#" * int(score * 20)
    print(f"  {label:<18} {score:.4f}  {bar}")
print()

changed = b.most_changed(baseline="clean")
print("most_changed(baseline='clean'):")
for label, score in changed:
    print(f"  {label:<18} {score:.4f}")

## 9. `tl.sweep` — one trace per replacement value

`tl.sweep(model, x, param, values, names=)` captures one intervened trace per value
and returns a `Bundle`. Here we sweep what flows through the relu layer.

In [ ]:
# _vocab: tl.sweep

import warnings

torch.manual_seed(42)
model_sw, x_sw = ZOO["tiny_mlp"]()

# Sweep the relu output: zeroed, half, unchanged
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # suppress UntypedStorage / RemovableHandle warnings
    sweep_bundle = tl.sweep(
        model_sw,
        x_sw,
        tl.func("relu"),  # intervention site: the relu output
        [torch.zeros(2, 8), x_sw * 0.5, x_sw],  # replacement tensors
        names=["zeroed", "half", "unmodified"],
    )

print("sweep_bundle names:", sweep_bundle.names)
print("is_structurally_consistent:", sweep_bundle.is_structurally_consistent)
print()
print("most_changed(baseline='unmodified'):")
for label, score in sweep_bundle.most_changed(baseline="unmodified"):
    print(f"  {label:<18} {score:.4f}")

## 10. Bundle save / load round-trip

In [ ]:
# _vocab: Bundle.save, tl.load

import pathlib

with tempfile.TemporaryDirectory() as td:
    bundle_path = pathlib.Path(td) / "clean_vs_ablated.tlspec"
    b.save(str(bundle_path))

    print("Saved bundle tree:")
    for p in sorted(bundle_path.rglob("*")):
        indent = "  " * (len(p.relative_to(bundle_path).parts))
        print(f"{indent}{p.name}")
    print()

    loaded_bundle = tl.load(str(bundle_path))
    print("loaded type :", type(loaded_bundle).__name__)
    print("loaded names:", loaded_bundle.names)

    # Verify alignment survived the round-trip
    print("\nPost-load most_changed(baseline='clean'):")
    for label, score in loaded_bundle.most_changed(baseline="clean"):
        print(f"  {label:<18} {score:.4f}")

## ⚠️ GAPs / ergonomic smells

- **`Bundle.__repr__` is the raw Python object string** (`<torchlens.intervention.bundle.Bundle object at 0x...>`).  
  There is no human-readable repr. `print(b)` tells a user nothing. A one-liner like  
  `Bundle(names=['clean','ablated'], layers=5, consistent=True)` would be much cleaner.
- **`tl.sweep` does NOT accept TorchLens intervention helpers** (e.g. `tl.scale(0.5)`,  
  `tl.zero_ablate()`, `tl.replace_with(...)`) as the `values` list — it expects raw tensors  
  or scalars. Passing an action helper raises `HookSignatureError`. Only tensor-valued sweeps  
  work today. A GAP: the `values` docstring says "Replacement values" but doesn't clarify  
  that TorchLens action objects are excluded.
- **`tl.fastlog.dry_run` does not accept `save=`** (the canonical unified predicate kwarg).  
  It takes the old `keep_op=` / `keep_module=` names only. Inconsistent with `tl.record(save=...)`.
- **`SuperOp.aggregate` rejects `statistic='mean'` with a position-arg call** (`sl.aggregate('mean')`  
  raises `ValueError: statistic must be one of...`). Must use keyword form `statistic='mean'`.  
  The positional form matches the signature but the internal validation rejects it — confusing error.
- **`Bundle.diff_pair(a, b)` returns a list of `(label, float)` tuples** while  
  `SuperOp.diff_pair(metric=)` returns a Tensor matrix. The same method name does different  
  things at bundle vs. layer level.
- **`Bundle.members` does not print cleanly** — it is a dict and prints the full Trace repr  
  for each value, making `print(b.members)` very verbose.